In [10]:
# 1. Google Drive 연결 및 SAS 원자료 불러오기 (팀원별 동일한 경로)
from pathlib import Path
import os, sys, subprocess, shutil
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Colab에서는 각자의 Google Drive만 연결합니다.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/hdac_data')
    IN_COLAB = True
except ImportError:
    # 로컬 검증용: SAS 파일 경로를 환경변수 KYRBS_SAS_PATH로 지정
    IN_COLAB = False
    DATA_DIR = Path(os.environ.get('KYRBS_DATA_DIR', '.'))

# 반드시 개인 Drive에 자기 원자료를 두세요.
DATA_PATH = Path(os.environ.get('KYRBS_SAS_PATH', str(DATA_DIR / 'kyrbs2025.sas7bdat')))
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'SAS 원자료를 찾을 수 없습니다: {DATA_PATH}\n'
        '본인 Google Drive의 내 드라이브/hdac_data/ 폴더에 '
        'kyrbs2025.sas7bdat 파일을 올리고 다시 실행하세요. '
        '압축파일(.zip)이 아닌 압축 해제한 .sas7bdat 파일이어야 합니다.'
    )

required = ['INT_SPWD', 'INT_SPWD_TM', 'E_S_RCRD', 'SEX', 'GRADE',
            'E_SES', 'M_STR', 'PA_TOT', 'F_BR', 'W', 'STRATA', 'CLUSTER']

# 교수님 예시와 동일한 pyreadstat.read_sas7bdat 방식
try:
    import pyreadstat
except ImportError:
    print('SAS 원자료 읽기 패키지 pyreadstat 설치 중...')
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyreadstat'])
    import pyreadstat

raw, meta = pyreadstat.read_sas7bdat(str(DATA_PATH), usecols=required)
missing_columns = sorted(set(required) - set(raw.columns))
if missing_columns:
    raise KeyError(f'원자료에 필수 변수가 없습니다: {missing_columns}\n실제 변수명: {raw.columns.tolist()}')

for v in ['INT_SPWD', 'INT_SPWD_TM', 'E_S_RCRD', 'SEX', 'GRADE',
          'E_SES', 'M_STR', 'PA_TOT', 'F_BR', 'W']:
    raw[v] = pd.to_numeric(raw[v], errors='coerce')
raw['STRATA'] = raw['STRATA'].astype(str).str.strip()
raw['CLUSTER'] = raw['CLUSTER'].astype(str).str.strip()
raw['PSU_KEY'] = raw['STRATA'] + '__' + raw['CLUSTER']

if raw['W'].isna().any() or (raw['W'] <= 0).any():
    raise ValueError('가중치 W에 결측 또는 0 이하 값이 있습니다. 원자료를 확인하세요.')
print('SAS 원자료 불러오기 완료:', DATA_PATH.name)
print('전체 관측수:', len(raw), '명 | 불러온 변수:', len(required), '개')
print('가중치:', 'W', '| 층화:', 'STRATA', '| 집락:', 'CLUSTER')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAS 원자료 불러오기 완료: kyrbs2025.sas7bdat
전체 관측수: 54170 명 | 불러온 변수: 12 개
가중치: W | 층화: STRATA | 집락: CLUSTER


In [ ]:
# 2. 노출변수 재코딩, 연구대상 및 도메인 표시변수
# 기존 문서의 정의: INT_SPWD=1(미사용)은 0분, INT_SPWD=2(사용)은 INT_SPWD_TM 사용
raw['phone_min'] = np.where(raw['INT_SPWD'].eq(1),0,raw['INT_SPWD_TM'])
raw['eligible_school'] = raw['GRADE'].between(1,6)
raw['exposure_ok'] = raw['phone_min'].ge(0) & raw['phone_min'].le(1440)
raw['outcome_ok'] = raw['E_S_RCRD'].isin([1,2,3,4,5])
raw['in_analysis'] = (raw['eligible_school'] & raw['exposure_ok'] & raw['outcome_ok']).astype(int)
# 0분은 첫 집단에, 1~119분은 둘째 집단에 정확히 넣도록 명시적으로 지정
raw['time_group'] = np.select([
    raw.phone_min.eq(0), (raw.phone_min>0)&(raw.phone_min<120),
    (raw.phone_min>=120)&(raw.phone_min<240),
    (raw.phone_min>=240)&(raw.phone_min<360),raw.phone_min>=360],
    ['미사용 (0분)','0분 초과~2시간 미만','2~4시간 미만','4~6시간 미만','6시간 이상'], default='정의 불가')
GROUPS=['전체','미사용 (0분)','0분 초과~2시간 미만','2~4시간 미만','4~6시간 미만','6시간 이상']

n0=len(raw)
n1=int(raw.eligible_school.sum())
n2=int((raw.eligible_school & raw.exposure_ok).sum())
n3=int(raw.in_analysis.sum())
flow=pd.DataFrame([
    ['1단계','전체 조사 참여자','-',n0],
    ['2단계','중·고등학생 확인',f'대상 미해당 {n0-n1:,}명',n1],
    ['3단계','주중 스마트폰 사용시간 정의 가능',f'노출 정의 불가 {n1-n2:,}명',n2],
    ['4단계','주관적 학업성적 유효 응답',f'결과 결측 {n2-n3:,}명',n3],
    ['최종','최종 분석 대상',f'총 {n0-n3:,}명 제외',n3],
],columns=['단계','선정 기준','제외 인원 및 이유','남은 인원'])

group_ns={'전체':n3}
for g in GROUPS[1:]:group_ns[g]=int(((raw.in_analysis==1)&(raw.time_group==g)).sum())
assert sum(group_ns[g] for g in GROUPS[1:])==n3,'집단 구분 합계 불일치'
assert np.isclose(raw.loc[raw.in_analysis.eq(1),'W'].gt(0).mean(),1),'가중치 0 또는 결측 확인 필요'
print('단계별 대상자 선정:');display(flow)
print('최종 5개 집단:');display(pd.Series(group_ns,name='인원').to_frame())
print('학업성적 결측자 중 미사용자:',int((raw.outcome_ok.eq(False)&raw.INT_SPWD.eq(1)&raw.exposure_ok).sum()))


단계별 대상자 선정:


,단계,선정 기준,제외 인원 및 이유,남은 인원
0,1단계,전체 조사 참여자,-,54170
1,2단계,중·고등학생 확인,대상 미해당 0명,54170
2,3단계,주중 스마트폰 사용시간 정의 가능,노출 정의 불가 0명,54170
3,4단계,주관적 학업성적 유효 응답,결과 결측 6명,54164
4,최종,최종 분석 대상,총 6명 제외,54164


최종 5개 집단:


,인원
전체,54164
미사용 (0분),1688
0분 초과~2시간 미만,4866
2~4시간 미만,18206
4~6시간 미만,16168
6시간 이상,13236


학업성적 결측자 중 미사용자: 0


In [ ]:
# 3. 설계기반 추정함수 (원자료의 모든 층·집락 유지)
# 비율과 평균은 도메인의 가중 합으로 산출하고, SE는 층화 내 PSU 기여도의 Taylor 선형화로 계산
psus=raw[['STRATA','PSU_KEY']].drop_duplicates()
nh=psus.groupby('STRATA').size()
if (nh<2).any():
    raise ValueError('PSU가 1개뿐인 층이 있습니다. 단독 PSU 처리 방식을 검토하세요.')
weights=raw.W.to_numpy(float)
strata=raw.STRATA.to_numpy()
psu=raw.PSU_KEY.to_numpy()
def mask_for_group(group):
    m=raw.in_analysis.eq(1).to_numpy()
    if group!='전체':m &= raw.time_group.eq(group).to_numpy()
    return m

def survey_mean_se(values, valid, group):
    vals=pd.to_numeric(pd.Series(values),errors='coerce').to_numpy(float)
    dom=mask_for_group(group)&np.asarray(valid,dtype=bool)&np.isfinite(vals)&np.isfinite(weights)
    d=dom.astype(float)
    denom=np.sum(weights*d)
    if denom<=0:return np.nan,np.nan
    mu=np.sum(weights*d*np.nan_to_num(vals,nan=0.0))/denom
    lin=weights*d*(np.nan_to_num(vals,nan=0.0)-mu)
    tmp=pd.DataFrame({'STRATA':strata,'PSU_KEY':psu,'linearized':lin})
    psu_total=tmp.groupby(['STRATA','PSU_KEY'],sort=False)['linearized'].sum().reset_index()
    sizes=psu_total.groupby('STRATA')['linearized'].transform('size').astype(float)
    centered=psu_total.linearized-psu_total.groupby('STRATA')['linearized'].transform('mean')
    variance=np.sum((sizes/(sizes-1))*centered.pow(2))/denom**2
    return float(mu),float(np.sqrt(max(variance,0.0)))

def weighted_pct(var, code, group):
    valid=raw[var].notna().to_numpy()
    indicator=raw[var].eq(code).astype(float)
    mu,se=survey_mean_se(indicator,valid,group)
    return mu*100,se*100

def unweighted_count(var,code,group):
    return int((mask_for_group(group)&raw[var].eq(code).to_numpy()).sum())

def unweighted_pct(var,code,group):
    den=int((mask_for_group(group)&raw[var].notna().to_numpy()).sum())
    return (100*unweighted_count(var,code,group)/den) if den else np.nan

variables=[
    ('주관적 학업성적','E_S_RCRD',[(1,'상'),(2,'중상'),(3,'중'),(4,'중하'),(5,'하')]),
    ('성별','SEX',[(1,'남학생'),(2,'여학생')]),
    ('학년','GRADE',[(1,'중1'),(2,'중2'),(3,'중3'),(4,'고1'),(5,'고2'),(6,'고3')]),
    ('경제상태','E_SES',[(1,'상'),(2,'중상'),(3,'중'),(4,'중하'),(5,'하')]),
    ('스트레스 인지','M_STR',[(1,'대단히 많이 느낀다'),(2,'많이 느낀다'),(3,'조금 느낀다'),(4,'별로 느끼지 않는다'),(5,'전혀 느끼지 않는다')]),
    ('신체활동 (원코드)','PA_TOT',[(j,f'코드 {j}') for j in range(1,9)]),
    ('아침식사 빈도 (원코드)','F_BR',[(j,f'코드 {j}') for j in range(1,9)]),
]
print('층 개수:',len(nh),'집락 개수:',len(psus),'최소 층별 집락:',int(nh.min()))


층 개수: 106 집락 개수: 796 최소 층별 집락: 2


In [ ]:
# 4. B-2 Table 1 (설계기반 가중 % 및 가중 평균 SE)
headers=['변수/범주']+[f'{g} (n={group_ns[g]:,})' for g in GROUPS]
b2_rows=[];b3_rows=[];comparisons=[];pct_checks=[]

def add_b2(label,vals):b2_rows.append([label]+vals)
def add_b3(label,vals):b3_rows.append([label]+vals)
add_b2('**대상자 수 (비가중 n)**',[f'**{group_ns[g]:,}**' for g in GROUPS])
add_b3('**대상자 수 (비가중 n)**',[f'**{group_ns[g]:,}**' for g in GROUPS])
means=[survey_mean_se(raw.phone_min,raw.phone_min.notna().to_numpy(),g) for g in GROUPS]
add_b2('**주중 스마트폰 사용시간(분), 가중 평균(SE)**',[f'{mu:.1f} ({se:.1f})' for mu,se in means])
add_b3('**주중 스마트폰 사용시간(분), 비가중 평균(SD)**',[f'{raw.loc[mask_for_group(g),"phone_min"].mean():.1f} ({raw.loc[mask_for_group(g),"phone_min"].std():.1f})' for g in GROUPS])
for section,var,cats in variables:
    add_b2(f'**{section}**',['']*len(GROUPS))
    add_b3(f'**{section}**',['']*len(GROUPS))
    for cat,label in cats:
        wres=[weighted_pct(var,cat,g)[0] for g in GROUPS]
        ns=[unweighted_count(var,cat,g) for g in GROUPS]
        ures=[unweighted_pct(var,cat,g) for g in GROUPS]
        add_b2(label,[f'{n:,} ({p:.1f}%)' for n,p in zip(ns,wres)])
        add_b3(label,[f'{n:,} ({p:.1f}%)' for n,p in zip(ns,ures)])
        for g,n,u,w in zip(GROUPS,ns,ures,wres):
            comparisons.append({'변수':section,'범주':label,'집단':g,'비가중 n':n,'비가중 %':u,'가중 %':w,'차이(%p)':w-u})
    misses=[int((mask_for_group(g)&raw[var].isna().to_numpy()).sum()) for g in GROUPS]
    add_b2('결측 (비가중 n)',[f'{n:,}' for n in misses])
    add_b3('결측 (비가중 n)',[f'{n:,}' for n in misses])
    for g in GROUPS:
        pct_sum=sum(weighted_pct(var,cat,g)[0] for cat,_ in cats)
        pct_checks.append({'변수':section,'집단':g,'가중 비율 합계(%)':pct_sum,'결측 n':int((mask_for_group(g)&raw[var].isna().to_numpy()).sum())})

b2=pd.DataFrame(b2_rows,columns=headers)
b3=pd.DataFrame(b3_rows,columns=headers)
comparison_df=pd.DataFrame(comparisons)
check_df=pd.DataFrame(pct_checks)
assert np.allclose(check_df['가중 비율 합계(%)'],100,atol=0.05), '가중 범주 합계가 100%가 아닙니다. 원코드 범위를 점검하세요.'
print('B-2 전체 Table 1:');display(b2)
print('B-3 전체 비가중 Table 1:');display(b3)
print('범주별 가중 비율 합계 점검:');display(check_df)


B-2 전체 Table 1:


,변수/범주,"전체 (n=54,164)","미사용 (0분) (n=1,688)","0분 초과~2시간 미만 (n=4,866)","2~4시간 미만 (n=18,206)","4~6시간 미만 (n=16,168)","6시간 이상 (n=13,236)"
0,**대상자 수 (비가중 n)**,"**54,164**","**1,688**","**4,866**","**18,206**","**16,168**","**13,236**"
1,"**주중 스마트폰 사용시간(분), 가중 평균(SE)**",264.6 (1.8),0.0 (0.0),67.1 (0.5),165.2 (0.3),277.9 (0.3),493.5 (1.8)
2,**주관적 학업성적**,,,,,,
3,상,"6,478 (12.2%)",382 (23.2%),"1,302 (27.9%)","2,632 (14.6%)","1,412 (8.9%)",750 (5.8%)
4,중상,"13,870 (25.9%)",382 (23.2%),"1,579 (32.5%)","5,598 (31.3%)","3,935 (24.6%)","2,376 (17.8%)"
5,중,"15,669 (28.9%)",457 (27.2%),"1,187 (24.5%)","5,420 (29.5%)","5,018 (31.3%)","3,587 (26.9%)"
6,중하,"12,865 (23.4%)",275 (15.4%),613 (11.7%),"3,517 (19.0%)","4,254 (25.9%)","4,206 (31.7%)"
7,하,"5,282 (9.6%)",192 (11.0%),185 (3.5%),"1,039 (5.6%)","1,549 (9.3%)","2,317 (17.8%)"
8,결측 (비가중 n),0,0,0,0,0,0
9,**성별**,,,,,,


B-3 전체 비가중 Table 1:


,변수/범주,"전체 (n=54,164)","미사용 (0분) (n=1,688)","0분 초과~2시간 미만 (n=4,866)","2~4시간 미만 (n=18,206)","4~6시간 미만 (n=16,168)","6시간 이상 (n=13,236)"
0,**대상자 수 (비가중 n)**,"**54,164**","**1,688**","**4,866**","**18,206**","**16,168**","**13,236**"
1,"**주중 스마트폰 사용시간(분), 비가중 평균(SD)**",265.5 (171.5),0.0 (0.0),67.0 (22.6),165.3 (33.0),278.2 (33.3),494.6 (165.4)
2,**주관적 학업성적**,,,,,,
3,상,"6,478 (12.0%)",382 (22.6%),"1,302 (26.8%)","2,632 (14.5%)","1,412 (8.7%)",750 (5.7%)
4,중상,"13,870 (25.6%)",382 (22.6%),"1,579 (32.4%)","5,598 (30.7%)","3,935 (24.3%)","2,376 (18.0%)"
5,중,"15,669 (28.9%)",457 (27.1%),"1,187 (24.4%)","5,420 (29.8%)","5,018 (31.0%)","3,587 (27.1%)"
6,중하,"12,865 (23.8%)",275 (16.3%),613 (12.6%),"3,517 (19.3%)","4,254 (26.3%)","4,206 (31.8%)"
7,하,"5,282 (9.8%)",192 (11.4%),185 (3.8%),"1,039 (5.7%)","1,549 (9.6%)","2,317 (17.5%)"
8,결측 (비가중 n),0,0,0,0,0,0
9,**성별**,,,,,,


범주별 가중 비율 합계 점검:


,변수,집단,가중 비율 합계(%),결측 n
0,주관적 학업성적,전체,100.0,0
1,주관적 학업성적,미사용 (0분),100.0,0
2,주관적 학업성적,0분 초과~2시간 미만,100.0,0
3,주관적 학업성적,2~4시간 미만,100.0,0
4,주관적 학업성적,4~6시간 미만,100.0,0
5,주관적 학업성적,6시간 이상,100.0,0
6,성별,전체,100.0,0
7,성별,미사용 (0분),100.0,0
8,성별,0분 초과~2시간 미만,100.0,0
9,성별,2~4시간 미만,100.0,0


In [ ]:
# 5. B-3: 가중 vs 비가중 차이 비교 (3개 대표 변수)
# 교수님 지침의 3개 변수: 학년, 경제상태, 주관적 학업성적.
# 각 변수에서 |가중%-비가중%|가 가장 큰 집단/범주를 자동 선택.
selected=[]
for name in ['학년','경제상태','주관적 학업성적']:
    subset=comparison_df[(comparison_df['변수']==name)&(comparison_df['집단']!='전체')]
    selected.append(subset.loc[subset['차이(%p)'].abs().idxmax()])
top3=pd.DataFrame(selected).reset_index(drop=True)
display(top3[['변수','집단','범주','비가중 n','비가중 %','가중 %','차이(%p)']].round(2))

print('최대 차이:',top3['차이(%p)'].abs().max())


,변수,집단,범주,비가중 n,비가중 %,가중 %,차이(%p)
0,학년,미사용 (0분),고3,390,23.10,25.78,2.67
1,경제상태,미사용 (0분),상,385,22.81,24.13,1.32
2,주관적 학업성적,0분 초과~2시간 미만,상,1302,26.76,27.86,1.11


최대 차이: 2.674742425085771
